# 00 Model Template

**How to use this template**

1. Make a copy and rename it, for example `02_logistic_regression.ipynb`.
2. Fill in `MEMBER` and `MODEL_NAME` below.
3. Replace the example model in section 3 with your own model.
4. Keep sections 4 and 5 as they are, so every model is evaluated the same way.

**Rules for fair comparison**

- Train on `train`, tune on `validation`.
- Only evaluate on `test` once, with your final settings.
- Log every experiment, including the ones that did badly.
- Your model must output a **probability for each class**, with columns in the order of `config.LABELS`.

## 1. Setup

In [ ]:
MEMBER = "Your name"
MODEL_NAME = "example_model"

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/michael-alu/nlp_language_technologies.git
    %cd nlp_language_technologies
    !pip install -q -r requirements.txt

if Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.append(os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd

from src import config
from src import data_loading
from src import evaluation
from src import plots
from src.experiment_log import log_experiment
from src.reproducibility import FINAL_RUN_SEEDS, set_seed

## 2. Load the data

In [ ]:
train = data_loading.load_train()

validation = data_loading.load_validation()

test = data_loading.load_test()

zindi_test = data_loading.load_zindi_test()

train_ids = data_loading.labels_to_ids(train["category"])

validation_ids = data_loading.labels_to_ids(validation["category"])

test_ids = data_loading.labels_to_ids(test["category"])

print("Labels in order:", config.LABELS)

print("Sizes:", len(train), len(validation), len(test))

## 3. Build and train your model

Replace this example with your own model. The example always predicts the class shares from the training data, so it is the worst sensible model. Any real model should beat it.

In [ ]:
from sklearn.dummy import DummyClassifier

settings = {"strategy": "prior"}


def build_model():
    return DummyClassifier(strategy=settings["strategy"])

In [ ]:
set_seed(config.RANDOM_SEED)

model = build_model()

model.fit(train["content"], train_ids)

validation_probabilities = model.predict_proba(validation["content"])

## 4. Evaluate on the validation set

Run this section after **every** experiment. Change `experiment_name` and `notes` each time so the log shows how your experiments progressed.

In [ ]:
experiment_name = "first_try"

notes = "Describe what you changed and why"

validation_scores = evaluation.compute_scores(validation_ids, validation_probabilities)

evaluation.print_classification_report(validation_ids, validation_probabilities)

log_experiment(MEMBER, MODEL_NAME, experiment_name, "validation", settings, validation_scores, notes)

validation_scores

In [ ]:
plots.plot_confusion_matrix(validation_ids, validation_probabilities, MODEL_NAME, "validation")

In [ ]:
plots.plot_roc_curves(validation_ids, validation_probabilities, MODEL_NAME, "validation")

**Neural models only:** record the loss and macro-F1 for every epoch during training, then plot them.

```python
history = {"train_loss": [...], "validation_loss": [...], "validation_macro_f1": [...]}
plots.plot_learning_curves(history, MODEL_NAME)
```

### 4b. Cross-validation (baselines only: logistic regression and Naive Bayes)

One validation split has only 2 `burudani` articles, so one mistake changes macro-F1 a lot. Cross-validation trains and scores the model 15 times (5 folds, repeated 3 times) on the train and validation data together, and reports the mean and standard deviation.

Run this for your main experiments, so you compare settings using the mean macro-F1 instead of one noisy number.

In [ ]:
from src.cross_validation import cross_validate

train_and_validation = data_loading.load_train_and_validation()

cross_validation_scores = cross_validate(build_model, train_and_validation)

log_experiment(MEMBER, MODEL_NAME, experiment_name, "cross_validation", settings, cross_validation_scores, notes)

### 4c. Neural models only: calibration

Neural networks are often overconfident, which hurts log loss (Guo et al., 2017). Save the raw outputs of your last layer **before softmax** (the logits), then find the temperature that gives the best validation log loss:

```python
from src import calibration

temperature = calibration.find_best_temperature(validation_logits, validation_ids)
test_probabilities = calibration.softmax(test_logits, temperature)
```

Log your scores both before and after calibration, so the report can show the difference.

## 5. Final evaluation on the test set

Only run this once, after you have chosen your best settings using the validation set.

**Neural models:** train your final model once for each seed in `FINAL_RUN_SEEDS` and log each run as `final_seed_<seed>`, so the report can show the mean and spread:

```python
for seed in FINAL_RUN_SEEDS:
    set_seed(seed)
    # build, train and predict with your final settings here
    test_scores = evaluation.compute_scores(test_ids, test_probabilities)
    log_experiment(MEMBER, MODEL_NAME, f"final_seed_{seed}", "test", settings, test_scores)
```

In [ ]:
test_probabilities = model.predict_proba(test["content"])

test_scores = evaluation.compute_scores(test_ids, test_probabilities)

evaluation.print_classification_report(test_ids, test_probabilities)

log_experiment(MEMBER, MODEL_NAME, "final", "test", settings, test_scores, "Final model on the test set")

test_scores

In [ ]:
plots.plot_confusion_matrix(test_ids, test_probabilities, MODEL_NAME, "test")

plots.plot_roc_curves(test_ids, test_probabilities, MODEL_NAME, "test")

In [ ]:
evaluation.save_misclassified_examples(test, test_ids, test_probabilities, MODEL_NAME)

In [ ]:
zindi_probabilities = model.predict_proba(zindi_test["content"])

evaluation.save_zindi_submission(zindi_test, zindi_probabilities, MODEL_NAME)

## 6. Save your results (Colab only)

Colab deletes files when the session ends. Download your results and add them to the repository under `results/`.

In [ ]:
if IN_COLAB:
    from google.colab import files

    !zip -r results.zip results
    
    files.download("results.zip")